# Y.Afisha Marketing Analysis

## Objective
Analyze user behavior, sales patterns and marketing spend for Y.Afisha to optimize investment allocation across acquisition channels.

## Data Sources
- `visits_log_us.csv` — website sessions (Jan 2017 - Dec 2018)
- `orders_log_us.csv` — completed orders
- `costs_us.csv` — marketing spend by acquisition source

In [79]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [80]:
# Function to clean column names by stripping whitespace, converting to lowercase, and replacing spaces with underscores
def clean_column_names(df):
    df.columns = (df.columns
                  .str.strip()
                  .str.lower()
                  .str.replace(' ', '_'))
    return df

## Dataset 1: Visits Log

In [81]:
# Load the visits log data
df_visits = pd.read_csv('./data/visits_log_us.csv')

In [82]:
# Display basic information about the DataFrame and its memory usage
df_visits.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 359400 entries, 0 to 359399
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   Device     359400 non-null  str   
 1   End Ts     359400 non-null  str   
 2   Source Id  359400 non-null  int64 
 3   Start Ts   359400 non-null  str   
 4   Uid        359400 non-null  uint64
dtypes: int64(1), str(3), uint64(1)
memory usage: 71.1 MB


In [83]:
# Check df head to understand the structure of the data
print(df_visits.head(6))

    Device               End Ts  Source Id             Start Ts  \
0    touch  2017-12-20 17:38:00          4  2017-12-20 17:20:00   
1  desktop  2018-02-19 17:21:00          2  2018-02-19 16:53:00   
2    touch  2017-07-01 01:54:00          5  2017-07-01 01:54:00   
3  desktop  2018-05-20 11:23:00          9  2018-05-20 10:59:00   
4  desktop  2017-12-27 14:06:00          3  2017-12-27 14:06:00   
5  desktop  2017-09-03 21:36:00          5  2017-09-03 21:35:00   

                    Uid  
0  16879256277535980062  
1    104060357244891740  
2   7459035603376831527  
3  16174680259334210214  
4   9969694820036681168  
5  16007536194108375387  


In [84]:
# Check options in the 'Device' column
print(df_visits['Device'].value_counts())

Device
desktop    262567
touch       96833
Name: count, dtype: int64


In [85]:
# Check options in the 'Source Id' column
print(df_visits['Source Id'].value_counts())

Source Id
4     101794
3      85610
5      66905
2      47626
1      34121
9      13277
10     10025
7         36
6          6
Name: count, dtype: int64


## Observations
- `Device` has only 2 unique values (`desktop`, `touch`) — should be converted to `category`
- `Source Id` has only 9 unique values — should be converted to `category`
- `Start Ts` / `End Ts` loaded as `str` — should be converted to `datetime`
- `Uid` is `uint64` — appropriate, no change needed
- Column names contain spaces and uppercase — will be standardized to `snake_case`
- Memory usage: 71.1 MB — expected to reduce after optimization

In [86]:
# Reload the data with optimized data types and parse date columns
df_visits = pd.read_csv(
    './data/visits_log_us.csv',
    dtype={'Device': 'category', 'Source Id': 'category'},
    parse_dates=['Start Ts', 'End Ts'])

In [87]:
# Display basic information about the DataFrame and its memory usage after optimization
df_visits.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 359400 entries, 0 to 359399
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   Device     359400 non-null  category      
 1   End Ts     359400 non-null  datetime64[us]
 2   Source Id  359400 non-null  category      
 3   Start Ts   359400 non-null  datetime64[us]
 4   Uid        359400 non-null  uint64        
dtypes: category(2), datetime64[us](2), uint64(1)
memory usage: 8.9 MB


In [88]:
# Clean column names to avoid issues
df_visits = clean_column_names(df_visits)

In [89]:
# Display the cleaned column names to verify the changes
print(df_visits.columns)

Index(['device', 'end_ts', 'source_id', 'start_ts', 'uid'], dtype='str')


In [90]:
# Check for duplicates
print(f"df_visits number of duplicate rows: {df_visits.duplicated().sum()}")

df_visits number of duplicate rows: 0


In [91]:
# Check for null values
print(f"Null values in each column:\n{df_visits.isnull().sum()}")

Null values in each column:
device       0
end_ts       0
source_id    0
start_ts     0
uid          0
dtype: int64


## Dataset 2: Orders

In [92]:
# Load the orders log data
df_orders = pd.read_csv('./data/orders_log_us.csv')

In [93]:
# Display basic information about the orders DataFrame and its memory usage
df_orders.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 50415 entries, 0 to 50414
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Buy Ts   50415 non-null  str    
 1   Revenue  50415 non-null  float64
 2   Uid      50415 non-null  uint64 
dtypes: float64(1), str(1), uint64(1)
memory usage: 4.0 MB


In [94]:
# Check df head to understand the structure of the data
print(df_orders.head(6))

                Buy Ts  Revenue                   Uid
0  2017-06-01 00:10:00    17.00  10329302124590727494
1  2017-06-01 00:25:00     0.55  11627257723692907447
2  2017-06-01 00:27:00     0.37  17903680561304213844
3  2017-06-01 00:29:00     0.55  16109239769442553005
4  2017-06-01 07:58:00     0.37  14200605875248379450
5  2017-06-01 08:43:00     0.18  10402394430196413321


## Observations
- `Buy Ts` loaded as `str` — should be converted to `datetime`
- Column names contain spaces and uppercase — will be standardized to `snake_case`
- `Revenue` is `float64` — appropriate for monetary values, no change needed
- `Uid` is `uint64` — appropriate, no change needed
- Memory usage: 4.0 MB — already small, minimal improvement expected after optimization

In [95]:
# Reload the orders data with optimized data types and parse date columns
df_orders = pd.read_csv(
    './data/orders_log_us.csv',
    parse_dates=['Buy Ts'])

In [96]:
# Display basic information about the DataFrame and its memory usage after optimization
df_orders.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 50415 entries, 0 to 50414
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   Buy Ts   50415 non-null  datetime64[us]
 1   Revenue  50415 non-null  float64       
 2   Uid      50415 non-null  uint64        
dtypes: datetime64[us](1), float64(1), uint64(1)
memory usage: 1.2 MB


In [97]:
# Clean column names to avoid issues
df_orders = clean_column_names(df_orders)

In [98]:
# Display the cleaned column names to verify the changes
print(df_orders.columns)

Index(['buy_ts', 'revenue', 'uid'], dtype='str')


In [99]:
# Check for duplicate rows
print(f"df_orders duplicate rows: {df_orders.duplicated().sum()}")

df_orders duplicate rows: 0


In [100]:
# Check for null values
print(f"Null values in each column:\n{df_orders.isnull().sum()}")

Null values in each column:
buy_ts     0
revenue    0
uid        0
dtype: int64


## Dataset 3: Marketing Costs

In [101]:
# Load the marketing costs data
df_marketing_costs = pd.read_csv('./data/costs_us.csv')

In [102]:
# Display basic information about the marketing costs DataFrame and its memory usage
df_marketing_costs.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 2542 entries, 0 to 2541
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   source_id  2542 non-null   int64  
 1   dt         2542 non-null   str    
 2   costs      2542 non-null   float64
dtypes: float64(1), int64(1), str(1)
memory usage: 186.3 KB


In [103]:
# Check sources in the 'source_id' column
print(df_marketing_costs['source_id'].value_counts())

source_id
5     364
1     363
2     363
3     363
4     363
9     363
10    363
Name: count, dtype: int64


In [104]:
# Check df head to understand the structure of the data
print(df_marketing_costs.head(6))

   source_id          dt  costs
0          1  2017-06-01  75.20
1          1  2017-06-02  62.25
2          1  2017-06-03  36.53
3          1  2017-06-04  55.00
4          1  2017-06-05  57.08
5          1  2017-06-06  40.39


## Observations
- `source_id` has only 7 unique values — should be converted to `category`
- `dt` loaded as `str` — should be converted to `datetime`
- Column names already in `snake_case` — `clean_column_names()` applied for consistency
- `costs` is `float64` — appropriate for monetary values, no change needed
- Memory usage: 183.3 KB — already small, minimal improvement expected after optimization

In [105]:
# Reload the marketing costs data with optimized data types and parse date columns
df_marketing_costs = pd.read_csv('./data/costs_us.csv', dtype={'source_id': 'category'},
                                 parse_dates=['dt'])

In [106]:
# Display basic information about the DataFrame and its memory usage after optimization
df_marketing_costs.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 2542 entries, 0 to 2541
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   source_id  2542 non-null   category      
 1   dt         2542 non-null   datetime64[us]
 2   costs      2542 non-null   float64       
dtypes: category(1), datetime64[us](1), float64(1)
memory usage: 43.0 KB


In [107]:
# Clean column names to avoid issues
df_marketing_costs = clean_column_names(df_marketing_costs)

In [108]:
# Display the cleaned column names to verify the changes
print(df_marketing_costs.columns)

Index(['source_id', 'dt', 'costs'], dtype='str')


In [109]:
# Check for duplicate rows
print(
    f"df_marketing_costs duplicate rows: {df_marketing_costs.duplicated().sum()}")

df_marketing_costs duplicate rows: 0


In [110]:
# Check for null values
print(f"Null values in each column:\n{df_marketing_costs.isnull().sum()}")

Null values in each column:
source_id    0
dt           0
costs        0
dtype: int64


## Data Preparation Summary


### Optimization Results
| Dataset | Rows | Before | After | Reduction |
|---|---|---|---|---|
| `df_visits` | 359,400 | 71.1 MB | 8.9 MB | -87% |
| `df_orders` | 50,415 | 4.0 MB | 1.2 MB | -70% |
| `df_marketing_costs` | 2,542 | 183.3 KB | 43.0 KB | -77% |

### Changes Applied
| Dataset | Column | Change |
|---|---|---|
| `df_visits` | `device` | `str` → `category` |
| `df_visits` | `source_id` | `int64` → `category` |
| `df_visits` | `start_ts`, `end_ts` | `str` → `datetime` |
| `df_orders` | `buy_ts` | `str` → `datetime` |
| `df_marketing_costs` | `source_id` | `int64` → `category` |
| `df_marketing_costs` | `dt` | `str` → `datetime` |
| All datasets | column names | standardized to `snake_case` |

### Cross-dataset Observations
- Source IDs `6` and `7` appear in `df_visits` (6 and 36 rows respectively) 
but have no entries in `df_marketing_costs`, suggesting these may represent 
organic or unpaid traffic channels such as direct access or word of mouth.
- No missing values or duplicates found in any dataset.